# Local the model
Download it from gradio and move it to the GPU

In [4]:
import torch
import gradio as gr
from threading import Thread
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer

MODEL_ID = "Qwen/Qwen3-14B"
DEVICE = "mps"

print(f"Loading {MODEL_ID} on {DEVICE}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
).to(DEVICE)
model.eval()
print("Loaded.")

Loading Qwen/Qwen3-14B on mps...


Loading weights: 100%|██████████| 443/443 [00:00<00:00, 6628.44it/s]


Loaded.


# generate() test

In [6]:
messages = [{"role": "user", "content": "What's 2+2?"}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=True)

inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
print(inputs)
out = model.generate(**inputs, max_new_tokens=200, do_sample=True, temperature=0.6, top_p=0.95, top_k=20)
print(tokenizer.decode(out[0], skip_special_tokens=True))


{'input_ids': tensor([[151644,    872,    198,   3838,    594,    220,     17,     10,     17,
             30, 151645,    198, 151644,  77091,    198]], device='mps:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='mps:0')}
user
What's 2+2?
assistant
<think>
Okay, the user asked "What's 2+2?" That's a basic math question. Let me make sure I answer correctly. In standard arithmetic, 2 plus 2 equals 4. But wait, maybe they're testing if I know different contexts where 2+2 might not be 4? Like in some non-standard systems or modulo arithmetic?

Hmm, but the question seems straightforward. The user might just want the simple answer. Unless there's a trick here. Let me think. In normal base 10, 2+2 is definitely 4. In binary, 2 is 10, so 10 + 10 is 100, which is 4 in decimal. So even in binary, it's still 4. What about other bases? In base 3, 2 is still 2, so 2+2 would be 4, but in base 3, 4 is represented as


# Forward pass without gradio

In [7]:
print(f"model: {model}")
with torch.inference_mode():
   outputs = model(**inputs, output_hidden_states=True)

   print(type(outputs))
   print("logits shape:", outputs.logits.shape)
   print("num hidden states:", len(outputs.hidden_states))
   print("each hidden state shape:", outputs.hidden_states[0].shape)

model: Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 5120)
    (layers): ModuleList(
      (0-39): 40 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=5120, out_features=5120, bias=False)
          (k_proj): Linear(in_features=5120, out_features=1024, bias=False)
          (v_proj): Linear(in_features=5120, out_features=1024, bias=False)
          (o_proj): Linear(in_features=5120, out_features=5120, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=5120, out_features=17408, bias=False)
          (up_proj): Linear(in_features=5120, out_features=17408, bias=False)
          (down_proj): Linear(in_features=17408, out_features=5120, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((5120,), eps=1e-06)
        (post_atten